# 6.3 Learned representations. CLIP

In the previous notebooks we described regions with **handcrafted descriptors**: we decided what to measure (circularity, Hu moments, histogram moments, GLCM features...). **Learned representations** take a different approach: a neural network learns from data which visual properties are useful, and maps the region to a feature vector $\mathbf{f} \in \mathbb{R}^D$, called **embedding**, which captures shape, appearance and semantic information at once (see Fig. 1). Similar regions tend to be close in this learned feature space.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/handcrafted_vs_learned.png" width="800"/>
    <figcaption>Fig. 1. Handcrafted descriptors are manually designed measurements, while learned representations are obtained by a neural network trained on data.</figcaption>
</center>$\\[5pt]$

In this notebook we will play with **CLIP**, one of the most popular models to obtain such embeddings:

- what CLIP is and how it was trained (<a href="#631">section 6.3.1</a>),
- describing a whole image with it (<a href="#632">section 6.3.2</a>),
- describing the regions of an image (<a href="#633">section 6.3.3</a>), and
- visualizing their embeddings in a feature space (<a href="#634">section 6.3.4</a>).

## Problem context - Recognizing objects in a kitchen

A company of home-assistant robots wants its robot to know which objects are in a kitchen. The robot has already detected some regions in the images of its camera (see Fig. 2), but it doesn't know what they are. Training a classifier for every possible object is not feasible, so they asked us to explore a model that already learned general-purpose visual representations: CLIP.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/kitchen_regions.png" width="600"/>
    <figcaption>Fig. 2. Kitchen image captured by the robot, with some of the regions it has detected: which objects are they?</figcaption>
</center>$\\[5pt]$

## 6.3.1 CLIP <a id="631"></a>

**[CLIP](https://openai.com/index/clip/)** (Contrastive Language-Image Pre-training) was published by OpenAI in 2021 (see the [paper](https://arxiv.org/abs/2103.00020)). It has two encoders that map their inputs to the **same** embedding space of dimension $D$:

- an **image encoder** (a Vision Transformer or a CNN), which returns the image embedding $\mathbf{f}_I \in \mathbb{R}^D$, and
- a **text encoder** (a Transformer), which returns the text embedding $\mathbf{f}_T \in \mathbb{R}^D$.

Both embeddings are normalized to unit length, so their dot product is the **cosine similarity** $s(I,T) = \mathbf{f}_I \cdot \mathbf{f}_T$: the more an image and a text mean the same thing, the higher their similarity.

**How was it trained?** With 400 million (image, text) pairs collected from the internet, e.g. photos and their captions, so no manual labels were needed. In each training step, CLIP takes a batch of $N$ pairs, encodes the $N$ images and the $N$ texts, and computes the $N \times N$ matrix of their similarities (see Fig. 3). Then, a **contrastive loss** increases the similarity of the $N$ matching pairs (the diagonal) and decreases that of the $N^2 - N$ non-matching ones.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/clip_training.gif" width="650"/>
    <figcaption>Fig. 3. Contrastive training of CLIP with a batch of $N$ (image, text) pairs: starting from random similarities, the similarity of the matching pairs (diagonal) is pushed up, and the rest down. The final values are real CLIP similarities of regions of the kitchen image used in this notebook.</figcaption>
</center>$\\[5pt]$

As a result, CLIP can classify images into classes that it was never explicitly trained on (**zero-shot classification**): we write a text for each class, e.g. "a photo of a dog", "a photo of a cat"..., and choose the class whose text embedding is the most similar to the image embedding. The similarities $s_k$ with the $K$ texts can be turned into probabilities with a softmax:

$$p_k = \frac{e^{100\, s_k}}{\sum_{j=1}^{K} e^{100\, s_j}}$$

where the factor 100 is the scale (inverse temperature) learned by CLIP during training.

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> CLIP in today's AI systems</font>  
> **Suggested prompt**  
> *Which current AI systems use CLIP or CLIP-like image-text encoders inside (e.g. text-to-image generators like Stable Diffusion, or multimodal LLMs), and what role do the embeddings play in each one?*

## Preparing our setup

We will use [OpenCLIP](https://github.com/mlfoundations/open_clip), an open-source implementation of CLIP that also provides the original weights trained by OpenAI. Let's install it:

In [ ]:
!pip3 install open_clip_torch

Now, let's import the required libraries and modules:

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 06. Region Description'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (15.0, 8.0)
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from sklearn.decomposition import PCA
import torch
import open_clip

images_path = './images/'

Next, we load the **ViT-B/32** version of CLIP: its image encoder is a Vision Transformer that splits the image into patches of $32 \times 32$ pixels, and its embeddings have $D = 512$ dimensions. The weights are downloaded the first time.

`create_model_and_transforms()` also returns the `preprocess` function, which prepares an image for the image encoder: it resizes its shorter side to 224 pixels, takes the **central** $224 \times 224$ crop, and normalizes it.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32-quickgelu', pretrained='openai', device=device)
tokenizer = open_clip.get_tokenizer('ViT-B-32-quickgelu')
model.eval()

print(preprocess)

### Initial data

The robot sent us the next image of a kitchen, `kitchen.jpg`. Let's take a look!

In [ ]:
image = cv2.cvtColor(cv2.imread(images_path + 'kitchen.jpg'), cv2.COLOR_BGR2RGB)

print('Image size:', image.shape)
plt.imshow(image)
plt.axis('off');

## 6.3.2 Describing a whole image <a id="632"></a>

Let's start by computing the embedding of the whole image, and use it to recognize which kind of room it shows.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Encoding images and texts with OpenCLIP</font>  
> OpenCLIP computes the embeddings with the `encode_image()` and `encode_text()` methods of the model, so let's see how they work!

> **Suggested prompt**  
> *I need to compute CLIP embeddings with the `open_clip` library in Python, using the `model`, `preprocess` and `tokenizer` returned by `open_clip.create_model_and_transforms()` and `open_clip.get_tokenizer()`. Do not write the full solution. Explain what `preprocess` expects as input and returns, how to build a batch of images with `torch.stack()`, what `encode_image()` and `encode_text()` return, why we use `torch.no_grad()`, and how to normalize the embeddings to unit length.*

### **<span style="color:green"><b><i>ASSIGNMENT 1: Embedding images and texts</i></b></span>**

**What to do?** Complete the functions `embed_images()` and `embed_texts()`, which return the **normalized** embeddings of a list of images and of a list of texts, respectively, as NumPy arrays of size $N \times D$.

Then, the given function `zero_shot()` uses them to compute the probability of each text label for each image, as explained in section 6.3.1.

In [ ]:
# Assignment 1
def embed_images(images):
    """ Compute the CLIP embeddings of a list of images.

        Args:
            images: list of RGB images (NumPy arrays)

        Returns:
            F: N x D array with the unit-length embeddings of the N images
    """
    # Preprocess the images and stack them in a batch
    batch = torch.stack([preprocess(Image.fromarray(None)) for image in images]).to(device)

    # Encode them (no gradients needed)
    with torch.no_grad():
        F = model.encode_image(None)

    # Normalize the embeddings to unit length
    F = F / None

    return F.cpu().numpy()


def embed_texts(texts):
    """ Compute the CLIP embeddings of a list of texts.

        Args:
            texts: list of strings

        Returns:
            F: N x D array with the unit-length embeddings of the N texts
    """
    # Tokenize the texts
    tokens = tokenizer(None).to(device)

    # Encode them (no gradients needed)
    with torch.no_grad():
        F = model.encode_text(None)

    # Normalize the embeddings to unit length
    F = F / None

    return F.cpu().numpy()

In [ ]:
def zero_shot(F, labels):
    """ Zero-shot classification of N embeddings into K text labels.

        Args:
            F: N x D array with the unit-length image embeddings
            labels: list of K labels (e.g. 'a kitchen')

        Returns:
            P: N x K array with the probability of each label for each embedding
    """
    T = embed_texts(['a photo of ' + label for label in labels])
    S = 100 * F @ T.T                                  # Scaled cosine similarities
    S = np.exp(S - S.max(axis=1, keepdims=True))       # Softmax (stable version)
    return S / S.sum(axis=1, keepdims=True)

You can use the next code to **test your functions**, classifying the kitchen image into some types of rooms:

In [ ]:
f = embed_images([None])
print('Embedding size:', f.shape)
print('Norm of the embedding:', round(float(np.linalg.norm(f)), 4))

rooms = ['a kitchen', 'a bedroom', 'a living room', 'a bathroom', 'an office', 'a street']
P = zero_shot(None, rooms)
for room, p in zip(rooms, P[0]):
    print(f'{room:15s} {p:.3f}')

<font color='blue'>**Expected output (similar to):**  </font>

```
Embedding size: (1, 512)
Norm of the embedding: 1.0
a kitchen       0.996
a bedroom       0.000
a living room   0.002
a bathroom      0.000
an office       0.001
a street        0.000
```

## 6.3.3 Describing regions <a id="633"></a>

CLIP worked fine for the whole image... but our robot needs to know **which objects** are in the kitchen and **where** they are. Here comes an important limitation: CLIP returns **a single embedding per input image**, which summarizes all of it (actually, only its central $224 \times 224$ crop). It does not provide an embedding for each region or segment of the image.

To describe regions with learned representations, there are two main options:

- **Cropping each region** obtained with a segmentation method (Chapter 5), e.g. its bounding box or the pixels of its mask returned by SAM (notebook 5.3), and computing the embedding of each crop. This way, segmentation tells us **where** the regions are, and the embedding **what** they are. It is simple, but it needs one forward pass per region, and the crop loses the context around the region.
- **Pooling the learned feature maps inside the region**: the backbone computes feature maps for the whole image, and the features inside the region are aggregated (e.g. averaged) into a single vector (see Fig. 4). This requires models that provide good dense (per-pixel) features.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/region_feature_pooling.png" width="800"/>
    <figcaption>Fig. 4. Region features from learned feature maps: the features $F_c(x,y)$ computed by a vision backbone are averaged inside the region $R$ to obtain its embedding $\mathbf{f} \in \mathbb{R}^D$.</figcaption>
</center>$\\[5pt]$

We will follow the first option. The robot has already segmented some regions of the image, given by their bounding boxes $(x, y, w, h)$ (the same format as the `bbox` of the masks returned by SAM in notebook 5.3). The function `crop_region()` crops a region and pads it with white pixels to make it square, so the central crop of `preprocess` does not cut it.

In [ ]:
regions = {
    'stool 1': (500, 566, 116, 174),
    'stool 2': (654, 566, 116, 174),
    'stool 3': (806, 566, 126, 174),
    'lamp 1': (560, 100, 76, 180),
    'lamp 2': (840, 100, 76, 180),
    'lamp 3': (170, 136, 50, 144),
    'fridge': (1040, 260, 70, 340),
    'rug': (170, 592, 294, 188),
    'dishwasher': (148, 505, 92, 215),
    'fruit stand': (30, 345, 120, 155),
    'kettle': (728, 405, 50, 45),
    'cabinets': (410, 220, 226, 160),
    'faucet': (540, 385, 45, 100),
}


def crop_region(image, box):
    """ Crop a region given by its bounding box (x, y, w, h) and pad it to a square. """
    x, y, w, h = box
    crop = image[y:y+h, x:x+w]
    size = max(w, h)
    square = np.full((size, size, 3), 255, dtype=np.uint8)
    square[(size-h)//2:(size-h)//2+h, (size-w)//2:(size-w)//2+w] = crop
    return square


crops = [crop_region(image, box) for box in regions.values()]

# Show the regions
image_boxes = image.copy()
for name, (x, y, w, h) in regions.items():
    cv2.rectangle(image_boxes, (x, y), (x+w, y+h), (255, 0, 0), 3)
    cv2.putText(image_boxes, name, (x, y-6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
plt.imshow(image_boxes)
plt.axis('off');

### **<span style="color:green"><b><i>ASSIGNMENT 2: Recognizing the regions</i></b></span>**

**What to do?** Compute the embeddings of the cropped regions, `F` (an $N \times D$ array), and use the `zero_shot()` function to recognize the object in each region among the given `objects`.

In [ ]:
# Assignment 2
F = embed_images(None)
print('Embeddings size:', F.shape)
print()

objects = ['a stool', 'a pendant lamp', 'a refrigerator', 'a rug', 'a dishwasher',
           'a fruit bowl', 'a kettle', 'kitchen cabinets', 'a faucet']
P = zero_shot(None, None)

for name, p in zip(regions, P):
    print(f'{name:12s} -> {objects[p.argmax()]:17s} ({p.max():.2f})')

<font color='blue'>**Expected output (similar to):**  </font>

```
Embeddings size: (13, 512)

stool 1      -> a stool           (0.97)
stool 2      -> a stool           (0.99)
stool 3      -> a stool           (1.00)
lamp 1       -> a pendant lamp    (1.00)
lamp 2       -> a pendant lamp    (1.00)
lamp 3       -> a pendant lamp    (1.00)
fridge       -> kitchen cabinets  (0.80)
rug          -> a rug             (1.00)
dishwasher   -> a refrigerator    (0.85)
fruit stand  -> a fruit bowl      (0.70)
kettle       -> a kettle          (0.70)
cabinets     -> kitchen cabinets  (0.99)
faucet       -> a faucet          (0.94)
```

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now, **answer the following questions:**

- Which regions are not recognized correctly? Why do you think that happens?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- What could we change (in the regions, in the labels...) to improve the results?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

## 6.3.4 Visualizing the feature space <a id="634"></a>

Each region is now described by an embedding $\mathbf{f} \in \mathbb{R}^{512}$. If CLIP learned a good representation, **similar regions should be close in this feature space**, like the feature vectors of the plate characters in notebook 6.1.

We cannot plot 512 dimensions, but we can project the embeddings to 2D with **Principal Component Analysis (PCA)**, which keeps the two directions along which the embeddings vary the most.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> PCA with scikit-learn</font>  
> scikit-learn implements PCA in the [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) class, so let's see how it works!

> **Suggested prompt**  
> *I need to project a set of N feature vectors of dimension 512 (a NumPy array of size N x 512) to 2D with the `PCA` class of scikit-learn, to plot them. Do not write the full solution. Explain what the `n_components` parameter means, what `fit_transform()` returns, and how much information is lost when keeping only 2 components.*

### **<span style="color:green"><b><i>ASSIGNMENT 3: Plotting the embeddings</i></b></span>**

**What to do?** Project the embeddings `F` to 2D with PCA. The given function `plot_feature_space()` shows each region at its projected position.

In [ ]:
def plot_feature_space(points, crops, names):
    """ Show the 2D projection of the embeddings with a thumbnail of each region. """
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.scatter(points[:,0], points[:,1], s=10, color='tab:blue')
    for (x, y), crop, name in zip(points, crops, names):
        thumbnail = OffsetImage(crop, zoom=55/crop.shape[0])
        ax.add_artist(AnnotationBbox(thumbnail, (x, y), frameon=True, pad=0.1))
        ax.annotate(name, (x, y), xytext=(0, -38), textcoords='offset points', ha='center', fontsize=9)
    ax.margins(0.15)
    ax.set_xlabel('First principal component')
    ax.set_ylabel('Second principal component')
    ax.set_title('CLIP embeddings of the regions (PCA projection)')

In [ ]:
# Assignment 3
points = PCA(n_components=None).fit_transform(None)
print('Projected embeddings size:', points.shape)

plot_feature_space(points, crops, list(regions.keys()))

<font color='blue'>**Expected output (similar to):**  </font>

<img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/clip_feature_space.png" width="550"/>

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now, **answer the following questions:**

- Are the regions of the same kind of object close to each other? Which ones?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Compare this feature space with the one of circularity and elongation in notebook 6.1. What are the main differences between both kinds of descriptors?

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> What CLIP embeddings capture</font>  
> **Suggested prompt**  
> *I have described regions of a kitchen image with CLIP embeddings, recognized them with zero-shot classification, and plotted their 2D PCA projection. Guide me, without giving me the answers, to reason what information these embeddings capture, why some regions were misrecognized, and which limitations CLIP has to describe the regions of an image.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

## Conclusion

Well done! In this short notebook you have learned:

- the difference between handcrafted descriptors and learned representations (embeddings),
- how CLIP was trained to map images and texts to the same embedding space, and how to use it for zero-shot classification,
- that CLIP returns one embedding per input image, so describing regions requires cropping them (or pooling learned feature maps), and
- how to visualize embeddings in 2D with PCA, where similar regions appear close to each other.

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>